In [1]:
import os
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = os.environ['PATH'] + r';C:\hadoop\bin'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, avg, max, min, sum,
    upper, lower, length, concat_ws, trim,
    rank, dense_rank, row_number, lead, lag,
    month, year, round as spark_round
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("RetailAnalytics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

BRONZE = r"C:\new_env\hexaware-sql\june_13\capstone_2\bronze"
SILVER = r"C:\new_env\hexaware-sql\june_13\capstone_2\silver"
GOLD   = r"C:\new_env\hexaware-sql\june_13\capstone_2\gold"

In [2]:
# 1
stores_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_2\stores.csv",
    header=True, inferSchema=True
)
stores_df.show(truncate=False)

+--------+--------------------+---------+-----------+-----------+------------+
|store_id|store_name          |city     |state      |store_type |manager_name|
+--------+--------------------+---------+-----------+-----------+------------+
|S101    |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S102    |Metro Mart Bangalore|Bangalore|Karnataka  |Supermarket|Priya Reddy |
|S103    |Metro Mart Mumbai   |Mumbai   |Maharashtra|Hypermarket|Amit Kumar  |
|S104    |Metro Mart Chennai  |Chennai  |Tamil Nadu |Supermarket|Sneha Patel |
|S105    |Metro Mart Delhi    |Delhi    |Delhi      |Hypermarket|Farhan Ali  |
|S106    |Metro Mart Pune     |Pune     |Maharashtra|Mini Store |Neha Singh  |
|S107    |Metro Mart Kochi    |Kochi    |Kerala     |Mini Store |Arjun Verma |
|S108    |Metro Mart Jaipur   |Jaipur   |Rajasthan  |Supermarket|Meera Nair  |
+--------+--------------------+---------+-----------+-----------+------------+



In [3]:
# 2
products_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_2\products.csv",
    header=True, inferSchema=True
)
products_df.show(truncate=False)

+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|category   |brand       |supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|P101      |Laptop      |Electronics|Lenovo      |S201       |65000     |
|P102      |Mobile      |Electronics|Samsung     |S202       |25000     |
|P103      |Television  |Electronics|LG          |S203       |45000     |
|P104      |Office Chair|Furniture  |Featherlite |S204       |7000      |
|P105      |Study Table |Furniture  |Urban Ladder|S204       |12000     |
|P106      |Shoes       |Fashion    |Nike        |S205       |4500      |
|P107      |Watch       |Fashion    |Fastrack    |S206       |8000      |
|P108      |Backpack    |Fashion    |Wildcraft   |S206       |2500      |
|P109      |Refrigerator|Electronics|Whirlpool   |S203       |38000     |
|P110      |Sofa        |Furniture  |Godrej      |S204       |32000     |
|P111      |Headphones  |Electronics|S

In [4]:
# 3
inventory_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_2\inventory.csv",
    header=True, inferSchema=True
)
inventory_df.show(truncate=False)

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|I1001       |S101    |P101      |10            |5            |2026-01-10 |
|I1002       |S101    |P102      |25            |10           |2026-01-10 |
|I1003       |S101    |P104      |3             |5            |2026-01-11 |
|I1004       |S102    |P101      |8             |5            |2026-01-12 |
|I1005       |S102    |P103      |5             |4            |2026-01-12 |
|I1006       |S103    |P105      |2             |5            |2026-01-13 |
|I1007       |S103    |P106      |30            |10           |2026-01-14 |
|I1008       |S104    |P107      |4             |5            |2026-01-15 |
|I1009       |S105    |P108      |50            |20           |2026-01-15 |
|I1010       |S106    |P109      |NULL          |6            |2026-01-16 |
|I1011      

In [5]:
# 4
sales_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_2\sales.csv",
    header=True, inferSchema=True
)
sales_df.show(truncate=False)

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
|SA1001 |S101    |P101      |2026-01-10|1            |65000      |UPI         |
|SA1002 |S101    |P102      |2026-01-10|2            |50000      |Card        |
|SA1003 |S102    |P101      |2026-01-11|1            |65000      |UPI         |
|SA1004 |S103    |P106      |2026-01-12|4            |18000      |Cash        |
|SA1005 |S104    |P107      |2026-01-12|1            |8000       |Card        |
|SA1006 |S105    |P108      |2026-01-13|5            |12500      |UPI         |
|SA1007 |S106    |P109      |2026-01-14|1            |38000      |Card        |
|SA1008 |S107    |P110      |2026-01-15|1            |32000      |UPI         |
|SA1009 |S108    |P120      |2026-01-15|2            |10000      |Cash        |
|SA1010 |S101    |P104      |2026-01-16|

In [6]:
# 5
suppliers_df = spark.read.option("multiline", "true").json(
    r"C:\new_env\hexaware-sql\june_13\capstone_2\suppliers.json"
)
suppliers_df.show(truncate=False)

+---------+---------------------------------+------+-----------+------------------------+
|city     |contact                          |rating|supplier_id|supplier_name           |
+---------+---------------------------------+------+-----------+------------------------+
|Hyderabad|{techsource@mail.com, 9876500011}|4.5   |S201       |TechSource India        |
|Bangalore|{mobileworld@mail.com, NULL}     |4.2   |S202       |MobileWorld Distributors|
|Mumbai   |{NULL, 9876500013}               |4.4   |S203       |HomeTech Supply         |
|Delhi    |{urban@mail.com, 9876500014}     |4.0   |S204       |Urban Furniture Co      |
|Pune     |{NULL, NULL}                     |3.8   |S205       |Fashion Direct          |
+---------+---------------------------------+------+-----------+------------------------+



In [7]:
# 6
stores_df.printSchema()
products_df.printSchema()
inventory_df.printSchema()
sales_df.printSchema()
suppliers_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: int

In [8]:
# 7
print("Stores    :", stores_df.count())
print("Products  :", products_df.count())
print("Inventory :", inventory_df.count())
print("Sales     :", sales_df.count())
print("Suppliers :", suppliers_df.count())

Stores    : 8
Products  : 12
Inventory : 12
Sales     : 15
Suppliers : 5


In [9]:
# 8
stores_df.write.mode("overwrite").parquet(BRONZE + r"\stores")
products_df.write.mode("overwrite").parquet(BRONZE + r"\products")
inventory_df.write.mode("overwrite").parquet(BRONZE + r"\inventory")
sales_df.write.mode("overwrite").parquet(BRONZE + r"\sales")
suppliers_df.write.mode("overwrite").parquet(BRONZE + r"\suppliers")
print("Bronze Parquet saved")

Bronze Parquet saved


In [10]:
# 9
products_df.filter(col('supplier_id').isNull()).show(truncate=False)

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|P112      |T-Shirt     |Fashion |Puma |NULL       |1500      |
+----------+------------+--------+-----+-----------+----------+



In [11]:
# 10
inventory_df.filter(col('stock_quantity').isNull()).show(truncate=False)

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|I1010       |S106    |P109      |NULL          |6            |2026-01-16 |
+------------+--------+----------+--------------+-------------+-----------+



In [12]:
# 11
sales_df.filter(col('sale_amount').isNull()).show(truncate=False)

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
|SA1011 |S102    |P103      |2026-01-17|1            |NULL       |UPI         |
+-------+--------+----------+----------+-------------+-----------+------------+



In [13]:
# 12
sales_df.filter(col('payment_mode').isNull()).show(truncate=False)

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
|SA1010 |S101    |P104      |2026-01-16|2            |14000      |NULL        |
+-------+--------+----------+----------+-------------+-----------+------------+



In [14]:
# 13
inventory_clean = inventory_df.fillna({'stock_quantity': 0})
inventory_clean.show(truncate=False)

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|I1001       |S101    |P101      |10            |5            |2026-01-10 |
|I1002       |S101    |P102      |25            |10           |2026-01-10 |
|I1003       |S101    |P104      |3             |5            |2026-01-11 |
|I1004       |S102    |P101      |8             |5            |2026-01-12 |
|I1005       |S102    |P103      |5             |4            |2026-01-12 |
|I1006       |S103    |P105      |2             |5            |2026-01-13 |
|I1007       |S103    |P106      |30            |10           |2026-01-14 |
|I1008       |S104    |P107      |4             |5            |2026-01-15 |
|I1009       |S105    |P108      |50            |20           |2026-01-15 |
|I1010       |S106    |P109      |0             |6            |2026-01-16 |
|I1011      

In [15]:
# 14
sales_clean = sales_df.fillna({'sale_amount': 0})
sales_clean.show(truncate=False)

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
|SA1001 |S101    |P101      |2026-01-10|1            |65000      |UPI         |
|SA1002 |S101    |P102      |2026-01-10|2            |50000      |Card        |
|SA1003 |S102    |P101      |2026-01-11|1            |65000      |UPI         |
|SA1004 |S103    |P106      |2026-01-12|4            |18000      |Cash        |
|SA1005 |S104    |P107      |2026-01-12|1            |8000       |Card        |
|SA1006 |S105    |P108      |2026-01-13|5            |12500      |UPI         |
|SA1007 |S106    |P109      |2026-01-14|1            |38000      |Card        |
|SA1008 |S107    |P110      |2026-01-15|1            |32000      |UPI         |
|SA1009 |S108    |P120      |2026-01-15|2            |10000      |Cash        |
|SA1010 |S101    |P104      |2026-01-16|

In [16]:
# 15
sales_clean = sales_df.fillna({
    'sale_amount': 0,
    'payment_mode': 'Not Provided'
})
sales_clean.show(truncate=False)

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
|SA1001 |S101    |P101      |2026-01-10|1            |65000      |UPI         |
|SA1002 |S101    |P102      |2026-01-10|2            |50000      |Card        |
|SA1003 |S102    |P101      |2026-01-11|1            |65000      |UPI         |
|SA1004 |S103    |P106      |2026-01-12|4            |18000      |Cash        |
|SA1005 |S104    |P107      |2026-01-12|1            |8000       |Card        |
|SA1006 |S105    |P108      |2026-01-13|5            |12500      |UPI         |
|SA1007 |S106    |P109      |2026-01-14|1            |38000      |Card        |
|SA1008 |S107    |P110      |2026-01-15|1            |32000      |UPI         |
|SA1009 |S108    |P120      |2026-01-15|2            |10000      |Cash        |
|SA1010 |S101    |P104      |2026-01-16|

In [17]:
# 16
products_clean = products_df.fillna({'supplier_id': 'UNKNOWN'})
products_clean.show(truncate=False)

+----------+------------+-----------+------------+-----------+----------+
|product_id|product_name|category   |brand       |supplier_id|unit_price|
+----------+------------+-----------+------------+-----------+----------+
|P101      |Laptop      |Electronics|Lenovo      |S201       |65000     |
|P102      |Mobile      |Electronics|Samsung     |S202       |25000     |
|P103      |Television  |Electronics|LG          |S203       |45000     |
|P104      |Office Chair|Furniture  |Featherlite |S204       |7000      |
|P105      |Study Table |Furniture  |Urban Ladder|S204       |12000     |
|P106      |Shoes       |Fashion    |Nike        |S205       |4500      |
|P107      |Watch       |Fashion    |Fastrack    |S206       |8000      |
|P108      |Backpack    |Fashion    |Wildcraft   |S206       |2500      |
|P109      |Refrigerator|Electronics|Whirlpool   |S203       |38000     |
|P110      |Sofa        |Furniture  |Godrej      |S204       |32000     |
|P111      |Headphones  |Electronics|S

In [18]:
# 17
products_clean = products_df.fillna({'supplier_id': 'UNKNOWN'}).withColumn(
    'data_quality_status',
    when(col('supplier_id').isNull(), 'Incomplete').otherwise('Complete')
)

inventory_clean = inventory_df.fillna({'stock_quantity': 0}).withColumn(
    'data_quality_status',
    when(col('stock_quantity').isNull(), 'Incomplete').otherwise('Complete')
)

sales_clean = sales_df.fillna({
    'sale_amount': 0,
    'payment_mode': 'Not Provided'
}).withColumn(
    'data_quality_status',
    when(
        col('sale_amount').isNull() | col('payment_mode').isNull(),
        'Incomplete'
    ).otherwise('Complete')
)

products_clean.select('product_id', 'data_quality_status').show()
inventory_clean.select('inventory_id', 'data_quality_status').show()
sales_clean.select('sale_id', 'data_quality_status').show()

+----------+-------------------+
|product_id|data_quality_status|
+----------+-------------------+
|      P101|           Complete|
|      P102|           Complete|
|      P103|           Complete|
|      P104|           Complete|
|      P105|           Complete|
|      P106|           Complete|
|      P107|           Complete|
|      P108|           Complete|
|      P109|           Complete|
|      P110|           Complete|
|      P111|           Complete|
|      P112|           Complete|
+----------+-------------------+

+------------+-------------------+
|inventory_id|data_quality_status|
+------------+-------------------+
|       I1001|           Complete|
|       I1002|           Complete|
|       I1003|           Complete|
|       I1004|           Complete|
|       I1005|           Complete|
|       I1006|           Complete|
|       I1007|           Complete|
|       I1008|           Complete|
|       I1009|           Complete|
|       I1010|           Complete|
|       I1011|  

In [19]:
# 18
products_clean.write.mode("overwrite").parquet(SILVER + r"\products")
inventory_clean.write.mode("overwrite").parquet(SILVER + r"\inventory")
sales_clean.write.mode("overwrite").parquet(SILVER + r"\sales")
print("Silver Parquet saved")

Silver Parquet saved


In [20]:
# 19
suppliers_flat = suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)
suppliers_flat.show(truncate=False)

+-----------+------------------------+---------+------+----------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone     |email               |
+-----------+------------------------+---------+------+----------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011|techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |NULL      |mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013|NULL                |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014|urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |NULL      |NULL                |
+-----------+------------------------+---------+------+----------+--------------------+



In [21]:
# 20
suppliers_flat.select('supplier_id', 'supplier_name', 'phone').show()

+-----------+--------------------+----------+
|supplier_id|       supplier_name|     phone|
+-----------+--------------------+----------+
|       S201|    TechSource India|9876500011|
|       S202|MobileWorld Distr...|      NULL|
|       S203|     HomeTech Supply|9876500013|
|       S204|  Urban Furniture Co|9876500014|
|       S205|      Fashion Direct|      NULL|
+-----------+--------------------+----------+



In [22]:
# 21
suppliers_flat.select('supplier_id', 'supplier_name', 'email').show()

+-----------+--------------------+--------------------+
|supplier_id|       supplier_name|               email|
+-----------+--------------------+--------------------+
|       S201|    TechSource India| techsource@mail.com|
|       S202|MobileWorld Distr...|mobileworld@mail.com|
|       S203|     HomeTech Supply|                NULL|
|       S204|  Urban Furniture Co|      urban@mail.com|
|       S205|      Fashion Direct|                NULL|
+-----------+--------------------+--------------------+



In [23]:
# 22
suppliers_flat = suppliers_flat.fillna({'phone': 'Not Provided'})
suppliers_flat.show(truncate=False)

+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013  |NULL                |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Not Provided|NULL                |
+-----------+------------------------+---------+------+------------+--------------------+



In [24]:
# 23
suppliers_flat = suppliers_flat.fillna({'email': 'Not Provided'})
suppliers_flat.show(truncate=False)

+-----------+------------------------+---------+------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|phone       |email               |
+-----------+------------------------+---------+------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |9876500013  |Not Provided        |
|S204       |Urban Furniture Co      |Delhi    |4.0   |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Not Provided|Not Provided        |
+-----------+------------------------+---------+------+------------+--------------------+



In [25]:
# 24
suppliers_flat.write.mode("overwrite").parquet(SILVER + r"\suppliers_flat")
print("Flattened suppliers saved")

Flattened suppliers saved


In [26]:
# 25
prod_suppliers = products_clean.drop('data_quality_status').join(
    suppliers_flat,
    on='supplier_id',
    how='left'
)
prod_suppliers.show(truncate=False)

+-----------+----------+------------+-----------+------------+----------+------------------------+---------+------+------------+--------------------+
|supplier_id|product_id|product_name|category   |brand       |unit_price|supplier_name           |city     |rating|phone       |email               |
+-----------+----------+------------+-----------+------------+----------+------------------------+---------+------+------------+--------------------+
|S201       |P101      |Laptop      |Electronics|Lenovo      |65000     |TechSource India        |Hyderabad|4.5   |9876500011  |techsource@mail.com |
|S202       |P102      |Mobile      |Electronics|Samsung     |25000     |MobileWorld Distributors|Bangalore|4.2   |Not Provided|mobileworld@mail.com|
|S203       |P103      |Television  |Electronics|LG          |45000     |HomeTech Supply         |Mumbai   |4.4   |9876500013  |Not Provided        |
|S204       |P104      |Office Chair|Furniture  |Featherlite |7000      |Urban Furniture Co      |De

In [27]:
# 26
inv_products = inventory_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status'),
    on='product_id',
    how='left'
)
inv_products.show(truncate=False)

+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|category   |brand       |supplier_id|unit_price|
+----------+------------+--------+--------------+-------------+-----------+------------+-----------+------------+-----------+----------+
|P101      |I1001       |S101    |10            |5            |2026-01-10 |Laptop      |Electronics|Lenovo      |S201       |65000     |
|P102      |I1002       |S101    |25            |10           |2026-01-10 |Mobile      |Electronics|Samsung     |S202       |25000     |
|P104      |I1003       |S101    |3             |5            |2026-01-11 |Office Chair|Furniture  |Featherlite |S204       |7000      |
|P101      |I1004       |S102    |8             |5            |2026-01-12 |Laptop      |Electronics|Lenovo      |S201       |65000     |
|P103      |I1005       |S102    |5      

In [28]:
# 27
sales_stores = sales_clean.drop('data_quality_status').join(
    stores_df,
    on='store_id',
    how='left'
)
sales_stores.show(truncate=False)

+--------+-------+----------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|store_name          |city     |state      |store_type |manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+
|S101    |SA1001 |P101      |2026-01-10|1            |65000      |UPI         |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S101    |SA1002 |P102      |2026-01-10|2            |50000      |Card        |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket|Rahul Sharma|
|S102    |SA1003 |P101      |2026-01-11|1            |65000      |UPI         |Metro Mart Bangalore|Bangalore|Karnataka  |Supermarket|Priya Reddy |
|S103    |SA1004 |P106      |2026-01-12|4            |18000      |Cash        |Metro Mart Mumbai   |Mumbai   |Ma

In [29]:
# 28
sales_products = sales_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status'),
    on='product_id',
    how='left'
)
sales_products.show(truncate=False)

+----------+-------+--------+----------+-------------+-----------+------------+------------+-----------+------------+-----------+----------+
|product_id|sale_id|store_id|sale_date |quantity_sold|sale_amount|payment_mode|product_name|category   |brand       |supplier_id|unit_price|
+----------+-------+--------+----------+-------------+-----------+------------+------------+-----------+------------+-----------+----------+
|P101      |SA1001 |S101    |2026-01-10|1            |65000      |UPI         |Laptop      |Electronics|Lenovo      |S201       |65000     |
|P102      |SA1002 |S101    |2026-01-10|2            |50000      |Card        |Mobile      |Electronics|Samsung     |S202       |25000     |
|P101      |SA1003 |S102    |2026-01-11|1            |65000      |UPI         |Laptop      |Electronics|Lenovo      |S201       |65000     |
|P106      |SA1004 |S103    |2026-01-12|4            |18000      |Cash        |Shoes       |Fashion    |Nike        |S205       |4500      |
|P107      |S

In [30]:
# 29
complete_df = (
    sales_clean.drop('data_quality_status')
    .join(stores_df, on='store_id', how='left')
    .join(products_clean.drop('data_quality_status'), on='product_id', how='left')
    .join(suppliers_flat, on='supplier_id', how='left')
)
complete_df.show(truncate=False)

+-----------+----------+--------+-------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+------------------------+---------+------+------------+--------------------+
|supplier_id|product_id|store_id|sale_id|sale_date |quantity_sold|sale_amount|payment_mode|store_name          |city     |state      |store_type |manager_name|product_name|category   |brand       |unit_price|supplier_name           |city     |rating|phone       |email               |
+-----------+----------+--------+-------+----------+-------------+-----------+------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+----------+------------------------+---------+------+------------+--------------------+
|S201       |P101      |S101    |SA1001 |2026-01-10|1            |65000      |UPI         |Metro Mart Hyderabad|Hyderabad|Telangana  |Supermarket

In [31]:
# 30
products_clean.drop('data_quality_status').join(
    suppliers_flat,
    on='supplier_id',
    how='left'
).filter(col('supplier_name').isNull()).show(truncate=False)

+-----------+----------+------------+-----------+---------+----------+-------------+----+------+-----+-----+
|supplier_id|product_id|product_name|category   |brand    |unit_price|supplier_name|city|rating|phone|email|
+-----------+----------+------------+-----------+---------+----------+-------------+----+------+-----+-----+
|S206       |P107      |Watch       |Fashion    |Fastrack |8000      |NULL         |NULL|NULL  |NULL |NULL |
|S206       |P108      |Backpack    |Fashion    |Wildcraft|2500      |NULL         |NULL|NULL  |NULL |NULL |
|S999       |P111      |Headphones  |Electronics|Sony     |3000      |NULL         |NULL|NULL  |NULL |NULL |
|UNKNOWN    |P112      |T-Shirt     |Fashion    |Puma     |1500      |NULL         |NULL|NULL  |NULL |NULL |
+-----------+----------+------------+-----------+---------+----------+-------------+----+------+-----+-----+



In [32]:
# 31
inventory_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status'),
    on='product_id',
    how='left'
).filter(col('product_name').isNull()).show(truncate=False)

+----------+------------+--------+--------------+-------------+-----------+------------+--------+-----+-----------+----------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+--------------+-------------+-----------+------------+--------+-----+-----------+----------+
|P120      |I1012       |S108    |12            |5            |2026-01-18 |NULL        |NULL    |NULL |NULL       |NULL      |
+----------+------------+--------+--------------+-------------+-----------+------------+--------+-----+-----------+----------+



In [33]:
# 32
sales_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status'),
    on='product_id',
    how='left'
).filter(col('product_name').isNull()).show(truncate=False)

+----------+-------+--------+----------+-------------+-----------+------------+------------+--------+-----+-----------+----------+
|product_id|sale_id|store_id|sale_date |quantity_sold|sale_amount|payment_mode|product_name|category|brand|supplier_id|unit_price|
+----------+-------+--------+----------+-------------+-----------+------------+------------+--------+-----+-----------+----------+
|P120      |SA1009 |S108    |2026-01-15|2            |10000      |Cash        |NULL        |NULL    |NULL |NULL       |NULL      |
+----------+-------+--------+----------+-------------+-----------+------------+------------+--------+-----+-----------+----------+



In [34]:
# 33
sales_clean.drop('data_quality_status').join(
    stores_df,
    on='store_id',
    how='left'
).filter(col('store_name').isNull()).show(truncate=False)

+--------+-------+----------+---------+-------------+-----------+------------+----------+----+-----+----------+------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_mode|store_name|city|state|store_type|manager_name|
+--------+-------+----------+---------+-------------+-----------+------------+----------+----+-----+----------+------------+
+--------+-------+----------+---------+-------------+-----------+------------+----------+----+-----+----------+------------+



In [35]:
# 34
inventory_clean = inventory_clean.withColumn(
    'stock_status',
    when(col('stock_quantity') <= col('reorder_level'), 'Reorder Required')
    .otherwise('Sufficient Stock')
)
inventory_clean.select('inventory_id', 'stock_quantity', 'reorder_level', 'stock_status').show()

+------------+--------------+-------------+----------------+
|inventory_id|stock_quantity|reorder_level|    stock_status|
+------------+--------------+-------------+----------------+
|       I1001|            10|            5|Sufficient Stock|
|       I1002|            25|           10|Sufficient Stock|
|       I1003|             3|            5|Reorder Required|
|       I1004|             8|            5|Sufficient Stock|
|       I1005|             5|            4|Sufficient Stock|
|       I1006|             2|            5|Reorder Required|
|       I1007|            30|           10|Sufficient Stock|
|       I1008|             4|            5|Reorder Required|
|       I1009|            50|           20|Sufficient Stock|
|       I1010|             0|            6|Reorder Required|
|       I1011|             1|            3|Reorder Required|
|       I1012|            12|            5|Sufficient Stock|
+------------+--------------+-------------+----------------+



In [36]:
# 35
products_clean = products_clean.withColumn(
    'price_category',
    when(col('unit_price') >= 50000, 'Premium')
    .when(col('unit_price') >= 10000, 'Standard')
    .otherwise('Budget')
)
products_clean.select('product_name', 'unit_price', 'price_category').show()

+------------+----------+--------------+
|product_name|unit_price|price_category|
+------------+----------+--------------+
|      Laptop|     65000|       Premium|
|      Mobile|     25000|      Standard|
|  Television|     45000|      Standard|
|Office Chair|      7000|        Budget|
| Study Table|     12000|      Standard|
|       Shoes|      4500|        Budget|
|       Watch|      8000|        Budget|
|    Backpack|      2500|        Budget|
|Refrigerator|     38000|      Standard|
|        Sofa|     32000|      Standard|
|  Headphones|      3000|        Budget|
|     T-Shirt|      1500|        Budget|
+------------+----------+--------------+



In [37]:
# 36
sales_clean = sales_clean.withColumn(
    'revenue_category',
    when(col('sale_amount') >= 50000, 'High Revenue')
    .when(col('sale_amount') >= 15000, 'Medium Revenue')
    .otherwise('Low Revenue')
)
sales_clean.select('sale_id', 'sale_amount', 'revenue_category').show()

+-------+-----------+----------------+
|sale_id|sale_amount|revenue_category|
+-------+-----------+----------------+
| SA1001|      65000|    High Revenue|
| SA1002|      50000|    High Revenue|
| SA1003|      65000|    High Revenue|
| SA1004|      18000|  Medium Revenue|
| SA1005|       8000|     Low Revenue|
| SA1006|      12500|     Low Revenue|
| SA1007|      38000|  Medium Revenue|
| SA1008|      32000|  Medium Revenue|
| SA1009|      10000|     Low Revenue|
| SA1010|      14000|     Low Revenue|
| SA1011|          0|     Low Revenue|
| SA1012|      12000|     Low Revenue|
| SA1013|      16000|  Medium Revenue|
| SA1014|       7500|     Low Revenue|
| SA1015|      25000|  Medium Revenue|
+-------+-----------+----------------+



In [38]:
# 37
sales_clean = sales_clean.withColumn('month', month(col('sale_date')))
sales_clean.select('sale_id', 'sale_date', 'month').show()

+-------+----------+-----+
|sale_id| sale_date|month|
+-------+----------+-----+
| SA1001|2026-01-10|    1|
| SA1002|2026-01-10|    1|
| SA1003|2026-01-11|    1|
| SA1004|2026-01-12|    1|
| SA1005|2026-01-12|    1|
| SA1006|2026-01-13|    1|
| SA1007|2026-01-14|    1|
| SA1008|2026-01-15|    1|
| SA1009|2026-01-15|    1|
| SA1010|2026-01-16|    1|
| SA1011|2026-01-17|    1|
| SA1012|2026-01-18|    1|
| SA1013|2026-02-01|    2|
| SA1014|2026-02-02|    2|
| SA1015|2026-02-03|    2|
+-------+----------+-----+



In [39]:
# 38
sales_clean = sales_clean.withColumn('year', year(col('sale_date')))
sales_clean.select('sale_id', 'sale_date', 'year').show()

+-------+----------+----+
|sale_id| sale_date|year|
+-------+----------+----+
| SA1001|2026-01-10|2026|
| SA1002|2026-01-10|2026|
| SA1003|2026-01-11|2026|
| SA1004|2026-01-12|2026|
| SA1005|2026-01-12|2026|
| SA1006|2026-01-13|2026|
| SA1007|2026-01-14|2026|
| SA1008|2026-01-15|2026|
| SA1009|2026-01-15|2026|
| SA1010|2026-01-16|2026|
| SA1011|2026-01-17|2026|
| SA1012|2026-01-18|2026|
| SA1013|2026-02-01|2026|
| SA1014|2026-02-02|2026|
| SA1015|2026-02-03|2026|
+-------+----------+----+



In [40]:
# 39
inv_with_value = (
    inventory_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'unit_price'), on='product_id')
    .withColumn('inventory_value', col('stock_quantity') * col('unit_price'))
)
inv_with_value.select('inventory_id', 'stock_quantity', 'unit_price', 'inventory_value').show()

+------------+--------------+----------+---------------+
|inventory_id|stock_quantity|unit_price|inventory_value|
+------------+--------------+----------+---------------+
|       I1001|            10|     65000|         650000|
|       I1002|            25|     25000|         625000|
|       I1003|             3|      7000|          21000|
|       I1004|             8|     65000|         520000|
|       I1005|             5|     45000|         225000|
|       I1006|             2|     12000|          24000|
|       I1007|            30|      4500|         135000|
|       I1008|             4|      8000|          32000|
|       I1009|            50|      2500|         125000|
|       I1010|             0|     38000|              0|
|       I1011|             1|     32000|          32000|
+------------+--------------+----------+---------------+



In [41]:
# 40
suppliers_flat = suppliers_flat.withColumn(
    'supplier_quality',
    when(col('rating') >= 4.5, 'Excellent')
    .when(col('rating') >= 4.0, 'Good')
    .otherwise('Average')
)
suppliers_flat.select('supplier_name', 'rating', 'supplier_quality').show()

+--------------------+------+----------------+
|       supplier_name|rating|supplier_quality|
+--------------------+------+----------------+
|    TechSource India|   4.5|       Excellent|
|MobileWorld Distr...|   4.2|            Good|
|     HomeTech Supply|   4.4|            Good|
|  Urban Furniture Co|   4.0|            Good|
|      Fashion Direct|   3.8|         Average|
+--------------------+------+----------------+



In [42]:
# 41
stores_df.groupBy('state').count().orderBy(col('count').desc()).show()

+-----------+-----+
|      state|count|
+-----------+-----+
|Maharashtra|    2|
|  Telangana|    1|
| Tamil Nadu|    1|
|  Karnataka|    1|
|      Delhi|    1|
|     Kerala|    1|
|  Rajasthan|    1|
+-----------+-----+



In [43]:
# 42
products_clean.groupBy('category').count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|    5|
|  Furniture|    3|
|    Fashion|    4|
+-----------+-----+



In [44]:
# 43
products_clean.groupBy('brand').count().show()

+------------+-----+
|       brand|count|
+------------+-----+
|          LG|    1|
|        Nike|    1|
|    Fastrack|    1|
|   Wildcraft|    1|
|   Whirlpool|    1|
|      Godrej|    1|
|        Sony|    1|
|      Lenovo|    1|
|     Samsung|    1|
| Featherlite|    1|
|Urban Ladder|    1|
|        Puma|    1|
+------------+-----+



In [45]:
# 44
inv_with_value.join(
    stores_df.select('store_id', 'store_name'), on='store_id'
).groupBy('store_name').sum('inventory_value').show()

+--------------------+--------------------+
|          store_name|sum(inventory_value)|
+--------------------+--------------------+
|    Metro Mart Delhi|              125000|
|    Metro Mart Kochi|               32000|
|Metro Mart Hyderabad|             1296000|
|Metro Mart Bangalore|              745000|
|   Metro Mart Mumbai|              159000|
|  Metro Mart Chennai|               32000|
|     Metro Mart Pune|                   0|
+--------------------+--------------------+



In [46]:
# 45
inv_with_value.join(
    products_clean.drop('data_quality_status').select('product_id', 'category'), on='product_id'
).groupBy('category').sum('inventory_value').show()

+-----------+--------------------+
|   category|sum(inventory_value)|
+-----------+--------------------+
|Electronics|             2020000|
|  Furniture|               77000|
|    Fashion|              292000|
+-----------+--------------------+



In [47]:
# 46
inventory_clean.filter(col('stock_status') == 'Reorder Required').count()

5

In [48]:
# 47
sales_clean.agg(sum('sale_amount').alias('total_revenue')).show()

+-------------+
|total_revenue|
+-------------+
|       373000|
+-------------+



In [49]:
# 48
sales_clean.drop('data_quality_status').join(
    stores_df.select('store_id', 'store_name'), on='store_id'
).groupBy('store_name').sum('sale_amount').show()

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|    Metro Mart Delhi|           20000|
|    Metro Mart Kochi|           32000|
|   Metro Mart Jaipur|           10000|
|Metro Mart Hyderabad|          154000|
|Metro Mart Bangalore|           65000|
|   Metro Mart Mumbai|           30000|
|  Metro Mart Chennai|           24000|
|     Metro Mart Pune|           38000|
+--------------------+----------------+



In [50]:
# 49
sales_clean.drop('data_quality_status').join(
    stores_df.select('store_id', 'city'), on='store_id'
).groupBy('city').sum('sale_amount').show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Hyderabad|          154000|
|Bangalore|           65000|
|     Pune|           38000|
|   Mumbai|           30000|
|  Chennai|           24000|
|    Delhi|           20000|
|    Kochi|           32000|
|   Jaipur|           10000|
+---------+----------------+



In [51]:
# 50
sales_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status').select('product_id', 'category'), on='product_id'
).groupBy('category').sum('sale_amount').show()

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|Electronics|          243000|
|  Furniture|           58000|
|    Fashion|           62000|
+-----------+----------------+



In [52]:
# 51
sales_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status').select('product_id', 'product_name'), on='product_id'
).groupBy('product_name').sum('sale_amount').show()

+------------+----------------+
|product_name|sum(sale_amount)|
+------------+----------------+
|Office Chair|           14000|
|    Backpack|           20000|
|      Laptop|          130000|
|      Mobile|           75000|
|  Television|               0|
| Study Table|           12000|
|       Shoes|           18000|
|       Watch|           24000|
|Refrigerator|           38000|
|        Sofa|           32000|
+------------+----------------+



In [53]:
# 52
sales_clean.groupBy('payment_mode').sum('sale_amount').show()

+------------+----------------+
|payment_mode|sum(sale_amount)|
+------------+----------------+
|        Card|          133000|
|         UPI|          190500|
|        Cash|           35500|
|Not Provided|           14000|
+------------+----------------+



In [54]:
# 53
sales_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status').select('product_id', 'product_name'), on='product_id'
).groupBy('product_name').sum('sale_amount').orderBy(col('sum(sale_amount)').desc()).show(1)

+------------+----------------+
|product_name|sum(sale_amount)|
+------------+----------------+
|      Laptop|          130000|
+------------+----------------+
only showing top 1 row


In [55]:
# 54
sales_clean.drop('data_quality_status').join(
    stores_df.select('store_id', 'store_name'), on='store_id'
).groupBy('store_name').sum('sale_amount').orderBy(col('sum(sale_amount)').desc()).show(1)

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|Metro Mart Hyderabad|          154000|
+--------------------+----------------+
only showing top 1 row


In [56]:
# 55
sales_clean.drop('data_quality_status').join(
    products_clean.drop('data_quality_status').select('product_id', 'category'), on='product_id'
).groupBy('category').sum('sale_amount').orderBy(col('sum(sale_amount)').desc()).show(1)

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|Electronics|          243000|
+-----------+----------------+
only showing top 1 row


In [57]:
# 56
w = Window.orderBy(col('sale_amount').desc())
(sales_clean
 .withColumn('revenue_rank', rank().over(w))
 .select('sale_id', 'product_id', 'sale_amount', 'revenue_rank')
 .show())

+-------+----------+-----------+------------+
|sale_id|product_id|sale_amount|revenue_rank|
+-------+----------+-----------+------------+
| SA1001|      P101|      65000|           1|
| SA1003|      P101|      65000|           1|
| SA1002|      P102|      50000|           3|
| SA1007|      P109|      38000|           4|
| SA1008|      P110|      32000|           5|
| SA1015|      P102|      25000|           6|
| SA1004|      P106|      18000|           7|
| SA1013|      P107|      16000|           8|
| SA1010|      P104|      14000|           9|
| SA1006|      P108|      12500|          10|
| SA1012|      P105|      12000|          11|
| SA1009|      P120|      10000|          12|
| SA1005|      P107|       8000|          13|
| SA1014|      P108|       7500|          14|
| SA1011|      P103|          0|          15|
+-------+----------+-----------+------------+



In [58]:
# 57
store_rev = (
    sales_clean.drop('data_quality_status')
    .join(stores_df.select('store_id', 'store_name'), on='store_id')
    .groupBy('store_name')
    .sum('sale_amount')
    .withColumnRenamed('sum(sale_amount)', 'total_revenue')
)
w = Window.orderBy(col('total_revenue').desc())
store_rev.withColumn('store_rank', rank().over(w)).show()

+--------------------+-------------+----------+
|          store_name|total_revenue|store_rank|
+--------------------+-------------+----------+
|Metro Mart Hyderabad|       154000|         1|
|Metro Mart Bangalore|        65000|         2|
|     Metro Mart Pune|        38000|         3|
|    Metro Mart Kochi|        32000|         4|
|   Metro Mart Mumbai|        30000|         5|
|  Metro Mart Chennai|        24000|         6|
|    Metro Mart Delhi|        20000|         7|
|   Metro Mart Jaipur|        10000|         8|
+--------------------+-------------+----------+



In [59]:
# 58
cat_rev = (
    sales_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'product_name', 'category'), on='product_id')
    .groupBy('product_name', 'category')
    .sum('sale_amount')
    .withColumnRenamed('sum(sale_amount)', 'total_revenue')
)
w = Window.partitionBy('category').orderBy(col('total_revenue').desc())
cat_rev.withColumn('cat_rank', rank().over(w)).show()

+------------+-----------+-------------+--------+
|product_name|   category|total_revenue|cat_rank|
+------------+-----------+-------------+--------+
|      Laptop|Electronics|       130000|       1|
|      Mobile|Electronics|        75000|       2|
|Refrigerator|Electronics|        38000|       3|
|  Television|Electronics|            0|       4|
|       Watch|    Fashion|        24000|       1|
|    Backpack|    Fashion|        20000|       2|
|       Shoes|    Fashion|        18000|       3|
|        Sofa|  Furniture|        32000|       1|
|Office Chair|  Furniture|        14000|       2|
| Study Table|  Furniture|        12000|       3|
+------------+-----------+-------------+--------+



In [60]:
# 59
cat_rev = (
    sales_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'product_name', 'category'), on='product_id')
    .groupBy('product_name', 'category')
    .sum('sale_amount')
    .withColumnRenamed('sum(sale_amount)', 'total_revenue')
)
w = Window.partitionBy('category').orderBy(col('total_revenue').desc())
(cat_rev
 .withColumn('cat_rank', rank().over(w))
 .filter(col('cat_rank') == 1)
 .show())

+------------+-----------+-------------+--------+
|product_name|   category|total_revenue|cat_rank|
+------------+-----------+-------------+--------+
|      Laptop|Electronics|       130000|       1|
|       Watch|    Fashion|        24000|       1|
|        Sofa|  Furniture|        32000|       1|
+------------+-----------+-------------+--------+



In [61]:
# 60
cat_rev = (
    sales_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'product_name', 'category'), on='product_id')
    .groupBy('product_name', 'category')
    .sum('sale_amount')
    .withColumnRenamed('sum(sale_amount)', 'total_revenue')
)
w = Window.partitionBy('category').orderBy(col('total_revenue').desc())
(cat_rev
 .withColumn('cat_rank', rank().over(w))
 .filter(col('cat_rank') <= 3)
 .show())

+------------+-----------+-------------+--------+
|product_name|   category|total_revenue|cat_rank|
+------------+-----------+-------------+--------+
|      Laptop|Electronics|       130000|       1|
|      Mobile|Electronics|        75000|       2|
|Refrigerator|Electronics|        38000|       3|
|       Watch|    Fashion|        24000|       1|
|    Backpack|    Fashion|        20000|       2|
|       Shoes|    Fashion|        18000|       3|
|        Sofa|  Furniture|        32000|       1|
|Office Chair|  Furniture|        14000|       2|
| Study Table|  Furniture|        12000|       3|
+------------+-----------+-------------+--------+



In [62]:
# 61
state_rev = (
    sales_clean.drop('data_quality_status')
    .join(stores_df.select('store_id', 'store_name', 'state'), on='store_id')
    .groupBy('store_name', 'state')
    .sum('sale_amount')
    .withColumnRenamed('sum(sale_amount)', 'total_revenue')
)
w = Window.partitionBy('state').orderBy(col('total_revenue').desc())
(state_rev
 .withColumn('state_rank', rank().over(w))
 .filter(col('state_rank') == 1)
 .show())

+--------------------+-----------+-------------+----------+
|          store_name|      state|total_revenue|state_rank|
+--------------------+-----------+-------------+----------+
|    Metro Mart Delhi|      Delhi|        20000|         1|
|Metro Mart Bangalore|  Karnataka|        65000|         1|
|    Metro Mart Kochi|     Kerala|        32000|         1|
|     Metro Mart Pune|Maharashtra|        38000|         1|
|   Metro Mart Jaipur|  Rajasthan|        10000|         1|
|  Metro Mart Chennai| Tamil Nadu|        24000|         1|
|Metro Mart Hyderabad|  Telangana|       154000|         1|
+--------------------+-----------+-------------+----------+



In [63]:
# 62
w = (
    Window
    .orderBy('sale_date')
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
(sales_clean
 .withColumn('running_revenue', sum('sale_amount').over(w))
 .select('sale_id', 'sale_date', 'sale_amount', 'running_revenue')
 .show())

+-------+----------+-----------+---------------+
|sale_id| sale_date|sale_amount|running_revenue|
+-------+----------+-----------+---------------+
| SA1001|2026-01-10|      65000|          65000|
| SA1002|2026-01-10|      50000|         115000|
| SA1003|2026-01-11|      65000|         180000|
| SA1004|2026-01-12|      18000|         198000|
| SA1005|2026-01-12|       8000|         206000|
| SA1006|2026-01-13|      12500|         218500|
| SA1007|2026-01-14|      38000|         256500|
| SA1008|2026-01-15|      32000|         288500|
| SA1009|2026-01-15|      10000|         298500|
| SA1010|2026-01-16|      14000|         312500|
| SA1011|2026-01-17|          0|         312500|
| SA1012|2026-01-18|      12000|         324500|
| SA1013|2026-02-01|      16000|         340500|
| SA1014|2026-02-02|       7500|         348000|
| SA1015|2026-02-03|      25000|         373000|
+-------+----------+-----------+---------------+



In [64]:
# 63
w = Window.orderBy('sale_date')
(sales_clean
 .withColumn('prev_sale', lag('sale_amount', 1).over(w))
 .select('sale_id', 'sale_date', 'sale_amount', 'prev_sale')
 .show())

+-------+----------+-----------+---------+
|sale_id| sale_date|sale_amount|prev_sale|
+-------+----------+-----------+---------+
| SA1001|2026-01-10|      65000|     NULL|
| SA1002|2026-01-10|      50000|    65000|
| SA1003|2026-01-11|      65000|    50000|
| SA1004|2026-01-12|      18000|    65000|
| SA1005|2026-01-12|       8000|    18000|
| SA1006|2026-01-13|      12500|     8000|
| SA1007|2026-01-14|      38000|    12500|
| SA1008|2026-01-15|      32000|    38000|
| SA1009|2026-01-15|      10000|    32000|
| SA1010|2026-01-16|      14000|    10000|
| SA1011|2026-01-17|          0|    14000|
| SA1012|2026-01-18|      12000|        0|
| SA1013|2026-02-01|      16000|    12000|
| SA1014|2026-02-02|       7500|    16000|
| SA1015|2026-02-03|      25000|     7500|
+-------+----------+-----------+---------+



In [65]:
# 64
w = Window.orderBy('sale_date')
(sales_clean
 .withColumn('next_sale', lead('sale_amount', 1).over(w))
 .select('sale_id', 'sale_date', 'sale_amount', 'next_sale')
 .show())

+-------+----------+-----------+---------+
|sale_id| sale_date|sale_amount|next_sale|
+-------+----------+-----------+---------+
| SA1001|2026-01-10|      65000|    50000|
| SA1002|2026-01-10|      50000|    65000|
| SA1003|2026-01-11|      65000|    18000|
| SA1004|2026-01-12|      18000|     8000|
| SA1005|2026-01-12|       8000|    12500|
| SA1006|2026-01-13|      12500|    38000|
| SA1007|2026-01-14|      38000|    32000|
| SA1008|2026-01-15|      32000|    10000|
| SA1009|2026-01-15|      10000|    14000|
| SA1010|2026-01-16|      14000|        0|
| SA1011|2026-01-17|          0|    12000|
| SA1012|2026-01-18|      12000|    16000|
| SA1013|2026-02-01|      16000|     7500|
| SA1014|2026-02-02|       7500|    25000|
| SA1015|2026-02-03|      25000|     NULL|
+-------+----------+-----------+---------+



In [66]:
# 65
w = Window.partitionBy('product_id').orderBy('sale_date')
(sales_clean
 .withColumn('prev_sale', lag('sale_amount', 1).over(w))
 .filter(col('sale_amount') > col('prev_sale'))
 .select('sale_id', 'product_id', 'sale_date', 'prev_sale', 'sale_amount')
 .show())

+-------+----------+----------+---------+-----------+
|sale_id|product_id| sale_date|prev_sale|sale_amount|
+-------+----------+----------+---------+-----------+
| SA1013|      P107|2026-02-01|     8000|      16000|
+-------+----------+----------+---------+-----------+



In [67]:
# 66
stores_df.createOrReplaceTempView("stores")
products_clean.drop('data_quality_status').createOrReplaceTempView("products")
inventory_clean.drop('data_quality_status').createOrReplaceTempView("inventory")
sales_clean.drop('data_quality_status').createOrReplaceTempView("sales")
suppliers_flat.createOrReplaceTempView("suppliers")
print("Temp views created")

Temp views created


In [68]:
# 67
spark.sql('SELECT * FROM sales').show(truncate=False)

+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
|sale_id|store_id|product_id|sale_date |quantity_sold|sale_amount|payment_mode|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
|SA1001 |S101    |P101      |2026-01-10|1            |65000      |UPI         |High Revenue    |1    |2026|
|SA1002 |S101    |P102      |2026-01-10|2            |50000      |Card        |High Revenue    |1    |2026|
|SA1003 |S102    |P101      |2026-01-11|1            |65000      |UPI         |High Revenue    |1    |2026|
|SA1004 |S103    |P106      |2026-01-12|4            |18000      |Cash        |Medium Revenue  |1    |2026|
|SA1005 |S104    |P107      |2026-01-12|1            |8000       |Card        |Low Revenue     |1    |2026|
|SA1006 |S105    |P108      |2026-01-13|5            |12500      |UPI         |Low Revenue     |1    |2026|
|SA1007 |S106    |P109      

In [69]:
# 68
spark.sql('SELECT category, COUNT(*) as total FROM products GROUP BY category').show()

+-----------+-----+
|   category|total|
+-----------+-----+
|Electronics|    5|
|  Furniture|    3|
|    Fashion|    4|
+-----------+-----+



In [70]:
# 69
spark.sql("""
    SELECT st.store_name, SUM(s.sale_amount) as revenue
    FROM sales s
    JOIN stores st ON s.store_id = st.store_id
    GROUP BY st.store_name
    ORDER BY revenue DESC
""").show()

+--------------------+-------+
|          store_name|revenue|
+--------------------+-------+
|Metro Mart Hyderabad| 154000|
|Metro Mart Bangalore|  65000|
|     Metro Mart Pune|  38000|
|    Metro Mart Kochi|  32000|
|   Metro Mart Mumbai|  30000|
|  Metro Mart Chennai|  24000|
|    Metro Mart Delhi|  20000|
|   Metro Mart Jaipur|  10000|
+--------------------+-------+



In [71]:
# 70
spark.sql("""
    SELECT st.city, SUM(s.sale_amount) as revenue
    FROM sales s
    JOIN stores st ON s.store_id = st.store_id
    GROUP BY st.city
    ORDER BY revenue DESC
""").show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Hyderabad| 154000|
|Bangalore|  65000|
|     Pune|  38000|
|    Kochi|  32000|
|   Mumbai|  30000|
|  Chennai|  24000|
|    Delhi|  20000|
|   Jaipur|  10000|
+---------+-------+



In [72]:
# 71
spark.sql("SELECT * FROM inventory WHERE stock_quantity <= reorder_level").show()

+------------+--------+----------+--------------+-------------+-----------+----------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|    stock_status|
+------------+--------+----------+--------------+-------------+-----------+----------------+
|       I1003|    S101|      P104|             3|            5| 2026-01-11|Reorder Required|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|Reorder Required|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|Reorder Required|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|Reorder Required|
|       I1011|    S107|      P110|             1|            3| 2026-01-17|Reorder Required|
+------------+--------+----------+--------------+-------------+-----------+----------------+



In [73]:
# 72
spark.sql("""
    SELECT s.*
    FROM sales s
    LEFT JOIN products p ON s.product_id = p.product_id
    WHERE p.product_name IS NULL
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|        Cash|     Low Revenue|    1|2026|
+-------+--------+----------+----------+-------------+-----------+------------+----------------+-----+----+



In [74]:
# 73
spark.sql("""
    SELECT p.product_id, p.product_name, p.supplier_id
    FROM products p
    LEFT JOIN suppliers s ON p.supplier_id = s.supplier_id
    WHERE s.supplier_name IS NULL
""").show()

+----------+------------+-----------+
|product_id|product_name|supplier_id|
+----------+------------+-----------+
|      P107|       Watch|       S206|
|      P108|    Backpack|       S206|
|      P111|  Headphones|       S999|
|      P112|     T-Shirt|    UNKNOWN|
+----------+------------+-----------+



In [75]:
# 74
spark.sql('SELECT product_id, SUM(sale_amount) as revenue FROM sales GROUP BY product_id ORDER BY revenue DESC LIMIT 5').show()

+----------+-------+
|product_id|revenue|
+----------+-------+
|      P101| 130000|
|      P102|  75000|
|      P109|  38000|
|      P110|  32000|
|      P107|  24000|
+----------+-------+



In [76]:
# 75
spark.sql('SELECT payment_mode, SUM(sale_amount) as revenue FROM sales GROUP BY payment_mode').show()

+------------+-------+
|payment_mode|revenue|
+------------+-------+
|        Card| 133000|
|         UPI| 190500|
|        Cash|  35500|
|Not Provided|  14000|
+------------+-------+



In [79]:
complete_df.drop("city").write.mode("overwrite").parquet(GOLD + r"\sales_complete")
print("Gold sales output saved")

Gold sales output saved


In [93]:
complete_df_clean.drop("city").write.mode("overwrite").partitionBy("year", "month").parquet(GOLD + r"\sales_partitioned")
print("Gold partitioned by year and month saved")

Gold partitioned by year and month saved


In [94]:
# 78
incremental_data = """sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_mode
SA1016,S101,P101,2026-03-01,1,65000,UPI
SA1017,S102,P102,2026-03-02,2,50000,Card
SA1018,S103,P106,2026-03-03,3,13500,Cash"""

with open(r"C:\new_env\hexaware-sql\june_13\capstone_2\sales_incremental.csv", "w") as f:
    f.write(incremental_data)
print("Incremental file created")

Incremental file created


In [95]:
# 79
sales_inc_df = spark.read.csv(
    r"C:\new_env\hexaware-sql\june_13\capstone_2\sales_incremental.csv",
    header=True, inferSchema=True
)
sales_inc_df.show()

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1016|    S101|      P101|2026-03-01|            1|      65000|         UPI|
| SA1017|    S102|      P102|2026-03-02|            2|      50000|        Card|
| SA1018|    S103|      P106|2026-03-03|            3|      13500|        Cash|
+-------+--------+----------+----------+-------------+-----------+------------+



In [96]:
# 80
sales_inc_clean = sales_inc_df.fillna({
    'sale_amount': 0,
    'payment_mode': 'Not Provided'
})
sales_inc_clean.write.mode("append").parquet(SILVER + r"\sales")
print("Incremental sales appended")

Incremental sales appended


In [97]:
# 81
sales_full = spark.read.parquet(SILVER + r"\sales")
product_revenue = (
    sales_full
    .join(products_clean.drop('data_quality_status').select('product_id', 'product_name'), on='product_id')
    .groupBy('product_id', 'product_name')
    .agg(sum('sale_amount').alias('total_revenue'))
    .orderBy(col('total_revenue').desc())
)
product_revenue.show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      P101|      Laptop|       260000|
|      P102|      Mobile|       175000|
|      P106|       Shoes|        45000|
|      P109|Refrigerator|        38000|
|      P110|        Sofa|        32000|
|      P107|       Watch|        24000|
|      P108|    Backpack|        20000|
|      P104|Office Chair|        14000|
|      P105| Study Table|        12000|
|      P103|  Television|            0|
+----------+------------+-------------+



In [98]:
# 82
sales_full = spark.read.parquet(SILVER + r"\sales")
store_revenue = (
    sales_full
    .join(stores_df.select('store_id', 'store_name'), on='store_id')
    .groupBy('store_id', 'store_name')
    .agg(sum('sale_amount').alias('total_revenue'))
    .orderBy(col('total_revenue').desc())
)
store_revenue.show()

+--------+--------------------+-------------+
|store_id|          store_name|total_revenue|
+--------+--------------------+-------------+
|    S101|Metro Mart Hyderabad|       284000|
|    S102|Metro Mart Bangalore|       165000|
|    S103|   Metro Mart Mumbai|        57000|
|    S106|     Metro Mart Pune|        38000|
|    S107|    Metro Mart Kochi|        32000|
|    S104|  Metro Mart Chennai|        24000|
|    S105|    Metro Mart Delhi|        20000|
|    S108|   Metro Mart Jaipur|        10000|
+--------+--------------------+-------------+



In [101]:
sales_full_clean.drop("city").write.mode("overwrite").partitionBy("year", "month").parquet(GOLD + r"\sales_partitioned")
print("Gold re-partitioned")

Gold re-partitioned


In [102]:
# 84
before = sales_df.count()
after = spark.read.parquet(SILVER + r"\sales").count()
print(f"Before incremental load : {before}")
print(f"After  incremental load : {after}")
print(f"New records added       : {after - before}")

Before incremental load : 15
After  incremental load : 21
New records added       : 6


In [103]:
# 85
gold_85 = (
    sales_clean.drop('data_quality_status')
    .join(stores_df, on='store_id', how='left')
    .groupBy('store_id', 'store_name', 'city', 'state')
    .agg(
        count('sale_id').alias('total_sales'),
        sum('sale_amount').alias('total_revenue')
    )
)
gold_85.show(truncate=False)
gold_85.write.mode("overwrite").parquet(GOLD + r"\store_performance")

+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|store_name          |city     |state      |total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|S103    |Metro Mart Mumbai   |Mumbai   |Maharashtra|2          |30000        |
|S104    |Metro Mart Chennai  |Chennai  |Tamil Nadu |2          |24000        |
|S105    |Metro Mart Delhi    |Delhi    |Delhi      |2          |20000        |
|S107    |Metro Mart Kochi    |Kochi    |Kerala     |1          |32000        |
|S101    |Metro Mart Hyderabad|Hyderabad|Telangana  |4          |154000       |
|S102    |Metro Mart Bangalore|Bangalore|Karnataka  |2          |65000        |
|S106    |Metro Mart Pune     |Pune     |Maharashtra|1          |38000        |
|S108    |Metro Mart Jaipur   |Jaipur   |Rajasthan  |1          |10000        |
+--------+--------------------+---------+-----------+-----------+-------------+



In [104]:
# 86
gold_86 = (
    sales_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'product_name', 'category', 'brand'), on='product_id')
    .groupBy('product_id', 'product_name', 'category', 'brand')
    .agg(
        sum('quantity_sold').alias('total_quantity_sold'),
        sum('sale_amount').alias('total_revenue')
    )
)
gold_86.show(truncate=False)
gold_86.write.mode("overwrite").parquet(GOLD + r"\product_performance")

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|category   |brand       |total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|P102      |Mobile      |Electronics|Samsung     |3                  |75000        |
|P107      |Watch       |Fashion    |Fastrack    |3                  |24000        |
|P108      |Backpack    |Fashion    |Wildcraft   |8                  |20000        |
|P110      |Sofa        |Furniture  |Godrej      |1                  |32000        |
|P101      |Laptop      |Electronics|Lenovo      |2                  |130000       |
|P103      |Television  |Electronics|LG          |1                  |0            |
|P104      |Office Chair|Furniture  |Featherlite |2                  |14000        |
|P105      |Study Table |Furniture  |Urban Ladder|1                  |12000        |
|P106      |Shoes       |Fashion    |Nike        |4              

In [105]:
# 87
gold_87 = (
    inventory_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'product_name'), on='product_id')
    .select('store_id', 'product_id', 'product_name', 'stock_quantity', 'reorder_level', 'stock_status')
)
gold_87.show(truncate=False)
gold_87.write.mode("overwrite").parquet(GOLD + r"\inventory_reorder")

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|stock_status    |
+--------+----------+------------+--------------+-------------+----------------+
|S101    |P101      |Laptop      |10            |5            |Sufficient Stock|
|S101    |P102      |Mobile      |25            |10           |Sufficient Stock|
|S101    |P104      |Office Chair|3             |5            |Reorder Required|
|S102    |P101      |Laptop      |8             |5            |Sufficient Stock|
|S102    |P103      |Television  |5             |4            |Sufficient Stock|
|S103    |P105      |Study Table |2             |5            |Reorder Required|
|S103    |P106      |Shoes       |30            |10           |Sufficient Stock|
|S104    |P107      |Watch       |4             |5            |Reorder Required|
|S105    |P108      |Backpack    |50            |20           |Sufficient Stock|
|S106    |P109      |Refrige

In [106]:
# 88
gold_88 = suppliers_flat.select(
    'supplier_id', 'supplier_name', 'city',
    'rating', 'supplier_quality', 'phone', 'email'
)
gold_88.show(truncate=False)
gold_88.write.mode("overwrite").parquet(GOLD + r"\supplier_quality")

+-----------+------------------------+---------+------+----------------+------------+--------------------+
|supplier_id|supplier_name           |city     |rating|supplier_quality|phone       |email               |
+-----------+------------------------+---------+------+----------------+------------+--------------------+
|S201       |TechSource India        |Hyderabad|4.5   |Excellent       |9876500011  |techsource@mail.com |
|S202       |MobileWorld Distributors|Bangalore|4.2   |Good            |Not Provided|mobileworld@mail.com|
|S203       |HomeTech Supply         |Mumbai   |4.4   |Good            |9876500013  |Not Provided        |
|S204       |Urban Furniture Co      |Delhi    |4.0   |Good            |9876500014  |urban@mail.com      |
|S205       |Fashion Direct          |Pune     |3.8   |Average         |Not Provided|Not Provided        |
+-----------+------------------------+---------+------+----------------+------------+--------------------+



In [107]:
# 89
gold_89 = (
    sales_clean.drop('data_quality_status')
    .join(products_clean.drop('data_quality_status').select('product_id', 'category'), on='product_id')
    .groupBy('category')
    .agg(
        count('product_id').alias('total_products'),
        sum('quantity_sold').alias('total_quantity_sold'),
        sum('sale_amount').alias('total_revenue')
    )
)
gold_89.show(truncate=False)
gold_89.write.mode("overwrite").parquet(GOLD + r"\category_revenue")

+-----------+--------------+-------------------+-------------+
|category   |total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|Electronics|6             |7                  |243000       |
|Fashion    |5             |15                 |62000        |
|Furniture  |3             |4                  |58000        |
+-----------+--------------+-------------------+-------------+



In [108]:
# 90
gold_90 = (
    sales_clean
    .groupBy('payment_mode')
    .agg(
        count('sale_id').alias('total_transactions'),
        sum('sale_amount').alias('total_revenue')
    )
)
gold_90.show(truncate=False)
gold_90.write.mode("overwrite").parquet(GOLD + r"\payment_mode_report")
print("All Gold Reports saved!")

+------------+------------------+-------------+
|payment_mode|total_transactions|total_revenue|
+------------+------------------+-------------+
|Card        |5                 |133000       |
|UPI         |6                 |190500       |
|Cash        |3                 |35500        |
|Not Provided|1                 |14000        |
+------------+------------------+-------------+

All Gold Reports saved!
